In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
BASE_DIR = Path.cwd()
RAW_PATH = BASE_DIR / "dataset" / "final_ebay_data.csv"
df = pd.read_csv(RAW_PATH)

df["Is_Rugged"] = df["Model"].astype(str).str.contains("Rugged|Toughbook", case=False, na=False).astype(int)
is_dual = df["Storage Type"].astype(str).str.contains(r"\+", na=False)
df["Total_Storage_GB"] = np.where(
    is_dual,
    df["SSD Capacity (GB)"] + df["Hard Drive Capacity (GB)"],
    df[["SSD Capacity (GB)", "Hard Drive Capacity (GB)"]].max(axis=1),
)

cat_cols = ["Brand", "Processor", "GPU", "Resolution Size", "OS", "Storage Type", "Condition"]
base_num = ["Screen Size (in)", "Ram Size (GB)", "SSD Capacity (GB)", "Processor Speed (GHz)", "Hard Drive Capacity (GB)"]

experiments = {
    "1. Baseline (Original 5 numeric)": base_num,
    "2. Replace SSD/HDD with Total_Storage_GB only": ["Screen Size (in)", "Ram Size (GB)", "Processor Speed (GHz)", "Total_Storage_GB"],
    "3. Baseline + Is_Rugged only": base_num + ["Is_Rugged"],
}

y = df["Price ($)"]

for name, num_cols in experiments.items():
    X = df[num_cols + cat_cols]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    preprocess = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])
    pipe = Pipeline([
        ("preprocess", preprocess),
        ("model", RandomForestRegressor(n_estimators=200, random_state=42)),
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    print(f"{name:48s} -> MAE: ${mean_absolute_error(y_test, pred):.2f} | R2: {r2_score(y_test, pred):.3f}")

1. Baseline (Original 5 numeric)                 -> MAE: $93.50 | R2: 0.733
2. Replace SSD/HDD with Total_Storage_GB only    -> MAE: $91.62 | R2: 0.729
3. Baseline + Is_Rugged only                     -> MAE: $129.89 | R2: 0.396
